# Evaluation framework and 7-day model comparison

Leave-one-year-out evaluation with purging, 30-day embargo, inner CV, isotonic calibration and MCC-optimal thresholds; logistic regression, random forest and XGBoost with three predictor sets at 7 days. Cells marked "(v2)" replace the earlier P4-1/P4-2 cells, which are kept for the record.

**Thesis:** Sections 3.6–3.9, 4.6 · Table 4.7 · Appendix C.6, C.7  
**Input:** features_*_t+7.csv  
**Output:** oof_p5/*, p5_results_det.csv

Runs on Google Colab with the folder layout described in the repository README (`/content/drive/MyDrive/Colab Notebooks/MHW_Project_Data/...`). The printed outputs below are from the original run used in the thesis.

#Phase 4

P4-1 — Evaluation framework

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, json, time, numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

PROC = '/content/drive/MyDrive/Colab Notebooks/MHW_Project_Data/processed'
OOF  = f'{PROC}/oof'; os.makedirs(OOF, exist_ok=True)
H = 7
groups = json.load(open(f'{PROC}/feature_groups.json'))
T = {k: pd.read_csv(f'{PROC}/features_{k}_t+{H}.csv', index_col=0, parse_dates=True) for k in ('det', 'raw')}
YEARS = sorted(T['det'].year.unique())

def train_mask(tab, test_years):
    """Training rows when `test_years` are held out: drop test years, rows whose t+7 target
    falls in a test year (purge), and the first 30 days after a test year (embargo)."""
    m = ~tab.year.isin(test_years) & ~tab.target_year.isin(test_years)
    after = tab.year.isin([y + 1 for y in test_years]) & (tab.index.dayofyear <= 30)
    return m & ~after

def mcc_from(tp, fp, fn, tn):
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return np.where(den > 0, (tp * tn - fp * fn) / np.where(den > 0, den, 1), 0.0)

def best_threshold(y, p):
    th = np.arange(0.02, 0.99, 0.01)
    P = p[None, :] >= th[:, None]
    tp = (P & (y == 1)).sum(1); fp = (P & (y == 0)).sum(1)
    fn = ((~P) & (y == 1)).sum(1); tn = ((~P) & (y == 0)).sum(1)
    return th[np.argmax(mcc_from(tp, fp, fn, tn))]

class Climatology:
    """P(MHW) for each calendar month, from the training years only."""
    def fit(self, X, y):
        self.p = pd.Series(y.values, index=X.index.month).groupby(level=0).mean(); return self
    def predict_proba(self, X):
        p = self.p.reindex(X.index.month).fillna(self.p.mean()).values
        return np.c_[1 - p, p]

def loyo(kind, make_model, cols, name, inner_folds=4, verbose=True):
    """Leave-one-year-out; threshold chosen inside each training fold by inner year-grouped CV."""
    tab, rows, t0 = T[kind], [], time.time()
    for y in YEARS:
        tr, te = tab[train_mask(tab, [y])], tab[tab.year == y]
        tr_years = np.array(sorted(tr.year.unique()))
        oof = pd.Series(np.nan, index=tr.index)
        for chunk in np.array_split(tr_years, inner_folds):
            itr = tr[train_mask(tr, list(chunk))]
            iva = tr[tr.year.isin(chunk)]
            oof.loc[iva.index] = make_model().fit(itr[cols], itr.y).predict_proba(iva[cols])[:, 1]
        ok = oof.notna()
        th = best_threshold(tr.y[ok].values, oof[ok].values)
        p = make_model().fit(tr[cols], tr.y).predict_proba(te[cols])[:, 1]
        rows.append(pd.DataFrame({'year': y, 'y': te.y.values, 'prob': p, 'pred': (p >= th).astype(int),
                                  'thr': th}, index=te.index))
    out = pd.concat(rows); out.to_csv(f'{OOF}/{name}_{kind}.csv')
    if verbose: print(f'{name} [{kind}] done in {time.time() - t0:.0f}s')
    return out

def persistence(kind):
    """MHW state knowable on day t (SST above threshold >= 5 consecutive days), carried to t+7."""
    tab = T[kind]
    p = tab.mhw_causal.astype(float)
    out = pd.DataFrame({'year': tab.year, 'y': tab.y, 'prob': p, 'pred': p.astype(int), 'thr': 0.5}, index=tab.index)
    out.to_csv(f'{OOF}/persistence_{kind}.csv'); return out

def metrics(o, clim=None):
    y, pr, p = o.y.values, o.pred.values, o.prob.values
    tp = int(((pr == 1) & (y == 1)).sum()); fp = int(((pr == 1) & (y == 0)).sum())
    fn = int(((pr == 0) & (y == 1)).sum()); tn = int(((pr == 0) & (y == 0)).sum())
    bs = np.mean((p - y) ** 2)
    r = dict(MCC=float(mcc_from(tp, fp, fn, tn)),
             F1=2 * tp / max(2 * tp + fp + fn, 1), CSI=tp / max(tp + fp + fn, 1),
             PR_AUC=average_precision_score(y, p) if y.any() else np.nan,
             TP=tp, FP=fp, FN=fn, TN=tn)
    if clim is not None:
        r['BSS'] = 1 - bs / np.mean((clim.prob.reindex(o.index).values - y) ** 2)
    return r

def year_counts(o):
    g = o.assign(tp=(o.pred == 1) & (o.y == 1), fp=(o.pred == 1) & (o.y == 0),
                 fn=(o.pred == 0) & (o.y == 1), tn=(o.pred == 0) & (o.y == 0)).groupby('year')
    return g[['tp', 'fp', 'fn', 'tn']].sum().reindex(YEARS).values

def bootstrap_mcc(o, ref=None, B=2000, seed=0):
    """Resample whole years: MCC 95% CI, and for (model - ref) the CI and P(diff <= 0)."""
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(YEARS), (B, len(YEARS)))
    m = mcc_from(*year_counts(o)[idx].sum(1).T)
    out = {'MCC_lo': np.percentile(m, 2.5), 'MCC_hi': np.percentile(m, 97.5)}
    if ref is not None:
        d = m - mcc_from(*year_counts(ref)[idx].sum(1).T)
        out.update(dMCC_lo=np.percentile(d, 2.5), dMCC_hi=np.percentile(d, 97.5), p_not_better=np.mean(d <= 0))
    return out

def report(results, kind, ref_name='persistence', clim_name='climatology', years=None):
    rows = {}
    for name, o in results.items():
        oo = o if years is None else o[o.year.isin(years)]
        r = metrics(oo, results[clim_name])
        if years is None:
            r.update(bootstrap_mcc(oo, None if name == ref_name else results[ref_name]))
        rows[name] = r
    return pd.DataFrame(rows).T

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


P4-2 — Benchmarks and an SST-only logistic regression, leave-one-year-out

In [ ]:
SST = groups['sst'] + groups['calendar']
logit = lambda: make_pipeline(StandardScaler(),
                              LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000))
R = {}
for kind in ('det', 'raw'):
    R[kind] = {'persistence': persistence(kind),
               'climatology': loyo(kind, Climatology, ['doy_sin'], 'climatology'),
               'logit_sst':   loyo(kind, logit, SST, 'logit_sst')}
    tab = report(R[kind], kind)
    tab.to_csv(f'{PROC}/p4_benchmarks_{kind}.csv')
    print(f'\n=== {kind}: leave-one-year-out, 1982-2025 ===')
    print(tab[['MCC', 'MCC_lo', 'MCC_hi', 'F1', 'CSI', 'PR_AUC', 'BSS', 'dMCC_lo', 'dMCC_hi']].round(3).to_string())
    print(f'--- {kind}: 2023-2025 only ---')
    print(report(R[kind], kind, years=[2023, 2024, 2025])[['MCC', 'F1', 'CSI', 'PR_AUC', 'BSS']].round(3).to_string())

climatology [det] done in 8s
logit_sst [det] done in 50s

=== det: leave-one-year-out, 1982-2025 ===
               MCC  MCC_lo  MCC_hi     F1    CSI  PR_AUC    BSS  dMCC_lo  dMCC_hi
persistence  0.265   0.167   0.344  0.296  0.173   0.151 -0.191      NaN      NaN
climatology  0.143   0.000   0.231  0.183  0.101   0.064  0.000   -0.265   -0.031
logit_sst    0.361   0.285   0.414  0.411  0.259   0.350 -1.252    0.048    0.148
--- det: 2023-2025 only ---
               MCC     F1    CSI  PR_AUC    BSS
persistence  0.135  0.183  0.101   0.106 -0.438
climatology  0.000  0.000  0.000   0.122  0.000
logit_sst    0.317  0.377  0.233   0.304 -2.128
climatology [raw] done in 2s
logit_sst [raw] done in 31s

=== raw: leave-one-year-out, 1982-2025 ===
               MCC  MCC_lo  MCC_hi     F1    CSI  PR_AUC    BSS  dMCC_lo  dMCC_hi
persistence  0.318   0.225   0.379  0.365  0.223   0.212 -0.151      NaN      NaN
climatology -0.324  -0.399  -0.238  0.043  0.022   0.084  0.000   -0.769   -0.478
logi

P4-3 — Gate: reproduce Section 4

In [ ]:
lab_persist = pd.read_csv(f'{PROC}/predictor_table_1982_2025.csv', index_col=0, parse_dates=True)
hgb = lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, class_weight='balanced', random_state=0)
print('\n=== Section 4 split (train 1982-2018, threshold on 2019-2022, test 2023-2025) ===')
for kind, lab in (('det', 'mhw_detrended'), ('raw', 'mhw_raw')):
    tab = T[kind]
    tr = tab[(tab.year <= 2018) & (tab.target_year <= 2018)]
    va = tab[(tab.year.between(2019, 2022)) & (tab.target_year <= 2022)]
    te = tab[tab.year >= 2023]
    y  = te.y.values
    mcc_bin = lambda pred: float(mcc_from(((pred == 1) & (y == 1)).sum(), ((pred == 1) & (y == 0)).sum(),
                                          ((pred == 0) & (y == 1)).sum(), ((pred == 0) & (y == 0)).sum()))
    line = {'share_test': te.y.mean(),
            'persist_label(t)': mcc_bin(lab_persist[lab].reindex(te.index).values.astype(int)),
            'persist_causal':   mcc_bin(te.mhw_causal.values)}
    for nm, mk in (('logit_sst', logit), ('hgb_sst', hgb)):
        m = mk().fit(tr[SST], tr.y)
        th = best_threshold(va.y.values, m.predict_proba(va[SST])[:, 1])
        line[nm] = mcc_bin((m.predict_proba(te[SST])[:, 1] >= th).astype(int))
    print(kind, {k: round(v, 2) for k, v in line.items()})
print('plan Section 4 for comparison: det persist 0.29, best SST-only 0.33 | raw persist 0.34, best SST-only 0.31')


=== Section 4 split (train 1982-2018, threshold on 2019-2022, test 2023-2025) ===
det {'share_test': np.float64(0.08), 'persist_label(t)': 0.29, 'persist_causal': 0.14, 'logit_sst': 0.34, 'hgb_sst': 0.16}
raw {'share_test': np.float64(0.43), 'persist_label(t)': 0.34, 'persist_causal': 0.22, 'logit_sst': 0.36, 'hgb_sst': 0.28}
plan Section 4 for comparison: det persist 0.29, best SST-only 0.33 | raw persist 0.34, best SST-only 0.31


P4-1 (v2) — Evaluation framework (replaces the old P4-1)

In [1]:
from google.colab import drive; drive.mount('/content/drive')
import os, json, time, numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score

PROC = '/content/drive/MyDrive/Colab Notebooks/MHW_Project_Data/processed'
OOF  = f'{PROC}/oof'; os.makedirs(OOF, exist_ok=True)
H = 7
groups = json.load(open(f'{PROC}/feature_groups.json'))
T = {k: pd.read_csv(f'{PROC}/features_{k}_t+{H}.csv', index_col=0, parse_dates=True) for k in ('det', 'raw')}
YEARS = sorted(T['det'].year.unique())

def train_mask(tab, test_years):
    """Drop test years, rows whose t+7 target is in a test year (purge), first 30 days after a test year (embargo)."""
    m = ~tab.year.isin(test_years) & ~tab.target_year.isin(test_years)
    after = tab.year.isin([y + 1 for y in test_years]) & (tab.index.dayofyear <= 30)
    return m & ~after

def mcc_from(tp, fp, fn, tn):
    tp, fp, fn, tn = (np.asarray(v, dtype=float) for v in (tp, fp, fn, tn))
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return np.where(den > 0, (tp * tn - fp * fn) / np.where(den > 0, den, 1), 0.0)

def best_threshold(y, p):
    th = np.unique(np.r_[np.arange(0.02, 0.99, 0.01), p])       # includes the calibrated step values
    th = th[(th > 0) & (th < 1)]
    P = p[None, :] >= th[:, None]
    tp = (P & (y == 1)).sum(1); fp = (P & (y == 0)).sum(1)
    fn = ((~P) & (y == 1)).sum(1); tn = ((~P) & (y == 0)).sum(1)
    m = mcc_from(tp, fp, fn, tn)
    return th[np.argmax(m)], m.max()

def loyo(kind, grid, cols, name, inner_folds=4, verbose=True):
    """Leave-one-year-out. Inside each training fold only:
    1) inner year-grouped CV picks the hyper-parameters (max MCC),
    2) isotonic calibration is fitted on the inner out-of-fold probabilities,
    3) the threshold is chosen on the calibrated out-of-fold probabilities.
    grid = list of (label, model_factory)."""
    tab, rows, t0 = T[kind], [], time.time()
    for y in YEARS:
        tr, te = tab[train_mask(tab, [y])], tab[tab.year == y]
        chunks = np.array_split(np.array(sorted(tr.year.unique())), inner_folds)
        best = None
        for lab, make in grid:
            oof = pd.Series(np.nan, index=tr.index)
            for ch in chunks:
                itr, iva = tr[train_mask(tr, list(ch))], tr[tr.year.isin(ch)]
                oof.loc[iva.index] = make().fit(itr[cols], itr.y).predict_proba(iva[cols])[:, 1]
            ok = oof.notna()
            score = best_threshold(tr.y[ok].values, oof[ok].values)[1]
            if best is None or score > best[0]: best = (score, lab, make, oof[ok], tr.y[ok])
        _, lab, make, oof, yo = best
        iso = IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=1).fit(oof.values, yo.values)
        th, _ = best_threshold(yo.values, iso.predict(oof.values))
        p = iso.predict(make().fit(tr[cols], tr.y).predict_proba(te[cols])[:, 1])
        rows.append(pd.DataFrame({'year': y, 'y': te.y.values, 'prob': p, 'pred': (p >= th).astype(int),
                                  'thr': th, 'params': lab}, index=te.index))
    out = pd.concat(rows); out.to_csv(f'{OOF}/{name}_{kind}.csv')
    if verbose: print(f'{name} [{kind}] done in {time.time() - t0:.0f}s | params chosen: '
                      f'{out.groupby("year").params.first().value_counts().to_dict()}')
    return out

def persistence(kind):
    """Honest persistence: MHW state knowable on day t (SST above threshold >= 5 consecutive days)."""
    tab = T[kind]; p = tab.mhw_causal.astype(float)
    out = pd.DataFrame({'year': tab.year, 'y': tab.y, 'prob': p, 'pred': p.astype(int), 'thr': 0.5}, index=tab.index)
    out.to_csv(f'{OOF}/persistence_{kind}.csv'); return out

def climatology(kind):
    """Brier-score reference only: P(MHW) by calendar month from the training years; no yes/no forecast."""
    tab, rows = T[kind], []
    for y in YEARS:
        tr, te = tab[train_mask(tab, [y])], tab[tab.year == y]
        pm = tr.y.groupby(tr.index.month).mean()
        rows.append(pd.DataFrame({'year': y, 'y': te.y.values, 'prob': pm.reindex(te.index.month).values,
                                  'pred': np.nan, 'thr': np.nan}, index=te.index))
    out = pd.concat(rows); out.to_csv(f'{OOF}/climatology_{kind}.csv'); return out

def onset_rows(o):
    """Forecast days whose target day t+7 is the first day of an MHW event."""
    prev = o.y.shift(1); consecutive = (o.index.to_series().diff().dt.days == 1).values
    return (o.y == 1) & (prev == 0) & consecutive

def metrics(o, clim):
    y, p = o.y.values, o.prob.values
    r = {'PR_AUC': average_precision_score(y, p) if y.any() else np.nan,
         'BSS': 1 - np.mean((p - y) ** 2) / np.mean((clim.prob.reindex(o.index).values - y) ** 2)}
    if o.pred.notna().all():
        pr = o.pred.values.astype(int)
        tp = int(((pr == 1) & (y == 1)).sum()); fp = int(((pr == 1) & (y == 0)).sum())
        fn = int(((pr == 0) & (y == 1)).sum()); tn = int(((pr == 0) & (y == 0)).sum())
        on = onset_rows(o)
        r.update(MCC=float(mcc_from(tp, fp, fn, tn)), F1=2 * tp / max(2 * tp + fp + fn, 1),
                 CSI=tp / max(tp + fp + fn, 1), onset_hit=float(o.pred[on].mean()), n_onsets=int(on.sum()),
                 TP=tp, FP=fp, FN=fn, TN=tn)
    return r

def year_counts(o):
    g = o.assign(tp=(o.pred == 1) & (o.y == 1), fp=(o.pred == 1) & (o.y == 0),
                 fn=(o.pred == 0) & (o.y == 1), tn=(o.pred == 0) & (o.y == 0)).groupby('year')
    return g[['tp', 'fp', 'fn', 'tn']].sum().reindex(YEARS).values

def bootstrap(o, refs, B=2000, seed=0):
    """Resample whole years (same resamples for every model): MCC CI and paired MCC-gain CI vs each reference."""
    idx = np.random.default_rng(seed).integers(0, len(YEARS), (B, len(YEARS)))
    m = mcc_from(*year_counts(o)[idx].sum(1).T)
    out = {'MCC_lo': np.percentile(m, 2.5), 'MCC_hi': np.percentile(m, 97.5)}
    for rn, ro in refs.items():
        d = m - mcc_from(*year_counts(ro)[idx].sum(1).T)
        out[f'gain_vs_{rn}_lo'], out[f'gain_vs_{rn}_hi'] = np.percentile(d, 2.5), np.percentile(d, 97.5)
    return out

def report(results, refs=('persistence', 'logit_sst'), years=None):
    """refs: honest persistence (plan rule) and the SST-only model (research question)."""
    rows = {}
    clim = results['climatology'] if years is None else results['climatology'][results['climatology'].year.isin(years)]
    for name, o in results.items():
        oo = o if years is None else o[o.year.isin(years)]
        r = metrics(oo, clim)
        if years is None and name != 'climatology':
            r.update(bootstrap(oo, {rn: results[rn] for rn in refs if rn in results and rn != name}))
        rows[name] = r
    return pd.DataFrame(rows).T

Mounted at /content/drive


P4-2 (v2) — Benchmarks rerun with the new framework

In [ ]:
SST = groups['sst'] + groups['calendar']
logit = lambda C: (lambda: make_pipeline(StandardScaler(),
                        LogisticRegression(C=C, class_weight='balanced', max_iter=3000)))
GRID_LOGIT = [(f'C={C}', logit(C)) for C in (0.01, 0.1, 1.0)]
R = {}
for kind in ('det', 'raw'):
    R[kind] = {'persistence': persistence(kind),
               'climatology': climatology(kind),
               'logit_sst':   loyo(kind, GRID_LOGIT, SST, 'logit_sst')}
    cols = ['MCC', 'MCC_lo', 'MCC_hi', 'F1', 'CSI', 'PR_AUC', 'BSS', 'onset_hit', 'gain_vs_persistence_lo', 'gain_vs_persistence_hi']
    tab = report(R[kind]); tab.to_csv(f'{PROC}/p4_benchmarks_v2_{kind}.csv')
    print(f'\n=== {kind}: leave-one-year-out 1982-2025 (n onsets = {int(tab.n_onsets.dropna().iloc[0])}) ===')
    print(tab.reindex(columns=cols).round(3).to_string())
    print(f'--- {kind}: 2023-2025 ---')
    print(report(R[kind], years=[2023, 2024, 2025]).reindex(columns=['MCC', 'F1', 'CSI', 'PR_AUC', 'BSS', 'onset_hit']).round(3).to_string())

logit_sst [det] done in 251s | params chosen: {'C=1.0': 29, 'C=0.01': 10, 'C=0.1': 5}

=== det: leave-one-year-out 1982-2025 (n onsets = 109) ===
               MCC  MCC_lo  MCC_hi     F1    CSI  PR_AUC    BSS  onset_hit  gain_vs_persistence_lo  gain_vs_persistence_hi
persistence  0.265   0.167   0.344  0.296  0.173   0.151 -0.191      0.092                     NaN                     NaN
climatology    NaN     NaN     NaN    NaN    NaN   0.064  0.000        NaN                     NaN                     NaN
logit_sst    0.363   0.284   0.419  0.413  0.260   0.337  0.180      0.174                   0.049                   0.148
--- det: 2023-2025 ---
               MCC     F1    CSI  PR_AUC    BSS  onset_hit
persistence  0.135  0.183  0.101   0.106 -0.438      0.000
climatology    NaN    NaN    NaN   0.122  0.000        NaN
logit_sst    0.311  0.372  0.229   0.273  0.098      0.222
logit_sst [raw] done in 276s | params chosen: {'C=0.01': 44}

=== raw: leave-one-year-out 1982-2025 (n 

#P5-0 — Load the framework

In [ ]:
OOF = f'{PROC}/oof_p5'; os.makedirs(OOF, exist_ok=True)

def best_threshold(y, p):
    """Threshold with the highest MCC. Candidates: a 0.01 grid plus the distinct calibrated values
    (or 300 upper quantiles when the probabilities are continuous)."""
    u = np.unique(p)
    extra = u if len(u) <= 500 else np.quantile(p, np.linspace(0.5, 1, 301))
    th = np.unique(np.r_[np.arange(0.01, 1.0, 0.01), extra])
    th = th[(th > 0) & (th < 1)]
    P = p[None, :] >= th[:, None]
    tp = (P & (y == 1)).sum(1); fp = (P & (y == 0)).sum(1)
    fn = ((~P) & (y == 1)).sum(1); tn = ((~P) & (y == 0)).sum(1)
    m = mcc_from(tp, fp, fn, tn)
    return th[np.argmax(m)], m.max()

P5-1 — Run all models (resumable)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

class XGBW:
    """XGBoost with class weighting from each training set (scale_pos_weight = negatives / positives)."""
    def __init__(self, **kw): self.kw = kw
    def fit(self, X, y):
        spw = float((y == 0).sum()) / max(float((y == 1).sum()), 1.0)
        self.m = XGBClassifier(scale_pos_weight=spw, subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                               tree_method='hist', n_jobs=-1, random_state=0, eval_metric='logloss', **self.kw)
        self.m.fit(X, y); return self
    def predict_proba(self, X): return self.m.predict_proba(X)

logit = lambda C: (lambda: make_pipeline(StandardScaler(),
                        LogisticRegression(C=C, class_weight='balanced', max_iter=3000)))
rf  = lambda leaf: (lambda: RandomForestClassifier(n_estimators=150, max_features='sqrt', min_samples_leaf=leaf,
                        max_samples=0.5, class_weight='balanced_subsample', n_jobs=-1, random_state=0))
xgb = lambda d: (lambda: XGBW(max_depth=d, learning_rate=0.05, n_estimators=300))
GRIDS = {'logit': [(f'C={C}', logit(C)) for C in (0.01, 0.1, 1.0)],
         'rf':    [(f'leaf={l}', rf(l)) for l in (10, 50)],
         'xgb':   [(f'depth={d}', xgb(d)) for d in (3, 5)]}
FSETS = {'sst':  groups['sst'] + groups['calendar'],
         'era5': groups['sst'] + groups['calendar'] + groups['era5'],
         'all':  groups['sst'] + groups['calendar'] + groups['era5'] + groups['indices']}

def run(kind, model, fset):
    """Resumable: a finished run is read back from processed/oof_p5 instead of being recomputed."""
    name = f'{model}_{fset}'
    f = f'{OOF}/{name}_{kind}.csv'
    if os.path.exists(f):
        return pd.read_csv(f, index_col=0, parse_dates=True)
    return loyo(kind, GRIDS[model], FSETS[fset], name, inner_folds=4 if model == 'logit' else 3)

KINDS = ['det']                    # primary target first; change to ['det', 'raw'] for the second pass
ORDER = [(m, s) for m in ('logit', 'xgb', 'rf') for s in ('sst', 'era5', 'all')]
P5 = {}
for kind in KINDS:
    P5[kind] = {'persistence': persistence(kind), 'climatology': climatology(kind)}
    for m, s in ORDER:
        P5[kind][f'{m}_{s}'] = run(kind, m, s)

P5-2 — Results and gate verdict

In [ ]:
def onset_year_counts(o):
    on = onset_rows(o)
    g = pd.DataFrame({'year': o.year[on], 'hit': o.pred[on]}).groupby('year').hit
    return g.sum().reindex(YEARS, fill_value=0).values, g.size().reindex(YEARS, fill_value=0).values

def compare(R, kind, B=2000, seed=0):
    idx = np.random.default_rng(seed).integers(0, len(YEARS), (B, len(YEARS)))
    names = [n for n in R if n != 'climatology']
    mcc = {n: mcc_from(*year_counts(R[n])[idx].sum(1).T) for n in names}
    ons = {}
    for n in names:
        h, c = onset_year_counts(R[n]); ons[n] = h[idx].sum(1) / np.maximum(c[idx].sum(1), 1)
    lo = lambda d: np.percentile(d, 2.5); hi = lambda d: np.percentile(d, 97.5)
    ci = lambda d: f'{lo(d):+.3f} to {hi(d):+.3f}'
    rows, verdict = {}, {}
    for n in names:
        r = metrics(R[n], R['climatology'])
        row = {'MCC': r['MCC'], 'MCC 95% CI': f'{lo(mcc[n]):.3f}-{hi(mcc[n]):.3f}', 'BSS': r['BSS'],
               'PR-AUC': r['PR_AUC'], 'onset hit': r['onset_hit']}
        if n != 'persistence':
            m, s = n.split('_')
            row['vs persistence'] = ci(mcc[n] - mcc['persistence'])
            if n != 'logit_sst':
                row['vs logit_sst'] = ci(mcc[n] - mcc['logit_sst'])
                verdict[(n, 'vs_logit_sst')] = lo(mcc[n] - mcc['logit_sst']) > 0
            base = {'era5': 'sst', 'all': 'era5'}.get(s)
            if base:
                d, do = mcc[n] - mcc[f'{m}_{base}'], ons[n] - ons[f'{m}_{base}']
                row['added group: MCC gain'] = ci(d)
                row['added group: onset-hit gain'] = ci(do)
                verdict[(n, 'added')] = lo(d) > 0
        rows[n] = row
    out = pd.DataFrame(rows).T
    out.to_csv(f'{PROC}/p5_results_{kind}.csv')
    return out, verdict

def decide(verdict):
    era5_types = [m for m in ('logit', 'xgb', 'rf') if verdict.get((f'{m}_era5', 'added'), False)]
    idx_types  = [m for m in ('logit', 'xgb', 'rf') if verdict.get((f'{m}_all', 'added'), False)]
    beats_ref  = [n for (n, k), v in verdict.items() if k == 'vs_logit_sst' and v and not n.endswith('_sst')]
    print(f'ERA5 adds skill (era5 vs sst, CI > 0) in: {era5_types or "none"}  -> needs >= 2 of 3')
    print(f'Indices add skill (all vs era5, CI > 0) in: {idx_types or "none"}  -> needs >= 2 of 3')
    print(f'ERA5/indices models beating logit_sst (CI > 0): {beats_ref or "none"}')
    print('GATE:', 'PASSED' if len(era5_types) >= 2 and beats_ref else 'NOT PASSED')

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 20)
for kind in P5:
    res, verdict = compare(P5[kind], kind)
    print(f'\n=================== {kind}: leave-one-year-out 1982-2025 ===================')
    num = ['MCC', 'BSS', 'PR-AUC', 'onset hit']
    print(res[['MCC', 'MCC 95% CI', 'BSS', 'PR-AUC', 'onset hit']].assign(**{c: res[c].astype(float).round(3) for c in num}).to_string())
    print('\n-- MCC gain, 95% CI (skill = whole interval above 0) --')
    print(res[['vs persistence', 'vs logit_sst', 'added group: MCC gain', 'added group: onset-hit gain']].fillna('').to_string())
    print()
    decide(verdict)
    print(f'\n-- {kind}: 2023-2025 only --')
    print(report(P5[kind], years=[2023, 2024, 2025]).reindex(columns=['MCC', 'BSS', 'PR_AUC', 'onset_hit']).round(3).to_string())


=================== det: leave-one-year-out 1982-2025 ===================
               MCC   MCC 95% CI    BSS  PR-AUC  onset hit
persistence  0.265  0.167-0.344 -0.191   0.151      0.092
logit_sst    0.361  0.281-0.418  0.180   0.338      0.174
logit_era5   0.365  0.289-0.427  0.197   0.372      0.193
logit_all    0.367  0.274-0.435  0.185   0.355      0.257
xgb_sst      0.312  0.241-0.364  0.138   0.286      0.248
xgb_era5     0.295  0.227-0.349  0.134   0.286      0.422
xgb_all      0.282  0.217-0.334  0.131   0.290      0.486
rf_sst       0.346  0.267-0.401  0.172   0.330      0.147
rf_era5      0.369  0.283-0.429  0.184   0.348      0.193
rf_all       0.360  0.278-0.423  0.181   0.348      0.193

-- MCC gain, 95% CI (skill = whole interval above 0) --
               vs persistence      vs logit_sst added group: MCC gain added group: onset-hit gain
persistence                                                                                      
logit_sst    +0.047 to +0.145     

In [ ]:
# ===== P5-3 (exploratory)  new-event skill: days with NO MHW ongoing on day t =====
kind = 'det'
quiet = T[kind].mhw_causal == 0                        # persistence says "no MHW" on these days
idx = np.random.default_rng(0).integers(0, len(YEARS), (2000, len(YEARS)))
sub = {n: o[quiet.reindex(o.index).values] for n, o in P5[kind].items() if n not in ('climatology', 'persistence')}
boot = {n: mcc_from(*year_counts(o)[idx].sum(1).T) for n, o in sub.items()}
rows = {}
for n, o in sub.items():
    y, p = o.y.values, o.pred.values.astype(int)
    tp, fp = int(((p == 1) & (y == 1)).sum()), int(((p == 1) & (y == 0)).sum())
    fn, tn = int(((p == 0) & (y == 1)).sum()), int(((p == 0) & (y == 0)).sum())
    m, s = n.split('_'); base = {'era5': 'sst', 'all': 'era5'}.get(s)
    d = boot[n] - boot[f'{m}_{base}'] if base else None
    rows[n] = {'new-event MCC': round(float(mcc_from(tp, fp, fn, tn)), 3),
               '95% CI': f'{np.percentile(boot[n], 2.5):.3f}-{np.percentile(boot[n], 97.5):.3f}',
               'hits': tp, 'false alarms': fp, 'misses': fn,
               'gain vs same model, previous set': '' if d is None else f'{np.percentile(d, 2.5):+.3f} to {np.percentile(d, 97.5):+.3f}'}
out = pd.DataFrame(rows).T.reindex([f'{m}_{s}' for m in ('logit', 'xgb', 'rf') for s in ('sst', 'era5', 'all')])
out.to_csv(f'{PROC}/p5_new_event_{kind}.csv')
print(f'{int(quiet.sum())} quiet days, of which {int(T[kind].y[quiet].sum())} have an MHW on t+7')
print(out.to_string())

15339 quiet days, of which 913 have an MHW on t+7
           new-event MCC       95% CI hits false alarms misses gain vs same model, previous set
logit_sst          0.285  0.222-0.327  323          724    590                                 
logit_era5          0.29  0.226-0.347  359          870    554                 -0.034 to +0.042
logit_all          0.295  0.218-0.354  347          784    566                 -0.027 to +0.034
xgb_sst            0.258  0.196-0.310  443         1627    470                                 
xgb_era5           0.241  0.183-0.289  547         2668    366                 -0.046 to +0.012
xgb_all            0.239  0.184-0.281  570         2906    343                 -0.035 to +0.030
rf_sst             0.273  0.213-0.320  330          825    583                                 
rf_era5            0.291  0.220-0.346  325          702    588                 -0.008 to +0.045
rf_all             0.279  0.212-0.331  325          767    588                 -0.029 

Check XGBOOST

In [2]:
# XGB check: is the 49% onset hit rate skill or more false alarms? (7-day lead, saved Phase 5 forecasts)
feat7 = pd.read_csv(f'{PROC}/features_det_t+7.csv', index_col=0, parse_dates=True)
quiet = feat7.mhw_causal == 0                              # days with no MHW knowable on the forecast day
R7 = {n: pd.read_csv(f'{PROC}/oof_p5/{n}_det.csv', index_col=0, parse_dates=True) for n in ['logit_all', 'xgb_all']}
idx = np.random.default_rng(0).integers(0, len(YEARS), (2000, len(YEARS)))
rows, B = {}, {}
for n, o in R7.items():
    q = o[quiet.reindex(o.index).fillna(False).values]
    y, p = o.y.values, o.pred.values.astype(int)
    tp, fp, fn = ((p == 1) & (y == 1)).sum(), ((p == 1) & (y == 0)).sum(), ((p == 0) & (y == 1)).sum()
    B[n] = mcc_from(*year_counts(q)[idx].sum(1).T)
    rows[n] = {'MCC': float(mcc_from(*year_counts(o).sum(0))), 'new-event MCC': float(mcc_from(*year_counts(q).sum(0))),
               'days forecast as MHW': int(p.sum()), 'hit rate': tp / (tp + fn), 'false alarm ratio': fp / (tp + fp)}
print(pd.DataFrame(rows).T.round(3))
d = B['xgb_all'] - B['logit_all']
print(f'new-event MCC, xgb_all minus logit_all: {np.percentile(d, 2.5):+.3f} to {np.percentile(d, 97.5):+.3f}')

             MCC  new-event MCC  days forecast as MHW  hit rate  \
logit_all  0.367          0.295                1629.0     0.492   
xgb_all    0.282          0.239                4053.0     0.685   

           false alarm ratio  
logit_all              0.640  
xgb_all                0.798  
new-event MCC, xgb_all minus logit_all: -0.105 to -0.002
